In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.regularizers import l2
import timm
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.optimizers import AdamW

In [ ]:
# Enable GPU usage
physical_devices = tf.config.list_physical_devices("GPU")
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print("Using GPU for training")
else:
    print("No GPU found, using CPU")

No GPU found, using CPU


In [ ]:
# Paths to dataset
train_csv_path = "train/train.csv"
train_images_path = "train/train/"
test_images_path = "test/test/"

# Load training CSV
train_df = pd.read_csv(train_csv_path)

In [6]:
print(train_df.head())

             ID                    Species  Usage
0   image_1.jpg           Eresus_annulatus  Train
1  image_2.jpeg        Agonum_sexpunctatum  Train
2  image_3.jpeg  Bembidion_quadrimaculatum  Train
3  image_4.jpeg               Nepa_cinerea  Train
4  image_5.jpeg          Carabus_nemoralis  Train


In [7]:
train_df.describe

<bound method NDFrame.describe of                    ID                    Species  Usage
0         image_1.jpg           Eresus_annulatus  Train
1        image_2.jpeg        Agonum_sexpunctatum  Train
2        image_3.jpeg  Bembidion_quadrimaculatum  Train
3        image_4.jpeg               Nepa_cinerea  Train
4        image_5.jpeg          Carabus_nemoralis  Train
...               ...                        ...    ...
8586  image_8588.jpeg         Dolycoris_baccarum  Train
8587  image_8589.jpeg          Carabus_nemoralis  Train
8588   image_8590.jpg           Eresus_annulatus  Train
8589  image_8591.jpeg       Stenodema_laevigatum  Train
8590  image_8592.jpeg            Carabus_auratus  Train

[8591 rows x 3 columns]>

In [ ]:
train_df["Species"].unique()

array(['Eresus_annulatus', 'Agonum_sexpunctatum',
       'Bembidion_quadrimaculatum', 'Nepa_cinerea', 'Carabus_nemoralis',
       'Carabus_granulatus', 'Lygus_pratensis', 'Carabus_monilis',
       'Carabus_convexus', 'Ilyocoris_cimicoides', 'Nabis_brevis',
       'Hydrometra_stagnorum', 'Carabus_auratus', 'Ischnodemus_sabuleti',
       'Agonum_muelleri', 'Graphosoma_lineatum', 'Bembidion_lampros',
       'Carabus_auronitens', 'Adelphocoris_seticornis',
       'Lygaeus_equestris', 'Carabus_coriaceus', 'Dolycoris_baccarum',
       'Corizus_hyoscyami', 'Stenodema_laevigatum', 'Lygocoris_pabulinus',
       'Leistus_ferrugineus', 'Carabus_problematicus',
       'Capsodes_gothicus', 'Agonum_marginatum', 'Nebria_brevicollis',
       'Rhinocoris_iracundus', 'Coreus_marginatus', 'Alydus_calcaratus',
       'Carpocoris_purpureipennis', 'Carabus_cancellatus',
       'Aelia_acuminata', 'Bembidion_tetracolum'], dtype=object)

In [ ]:
temp = pd.DataFrame()
temp = train_df["Species"].value_counts().reset_index()
temp.columns = ["Species", "Numbers"]

In [ ]:
import matplotlib.pyplot as plt

# Plot bar chart
plt.figure(figsize=(12, 5))
plt.bar(temp["Species"], temp["Numbers"], color="skyblue", edgecolor="black")

# Add labels and title
plt.xlabel("Species")
plt.ylabel("Count")
plt.title("Count of Each Species")
plt.xticks(rotation=45)

# Show plot
plt.show()

In [ ]:
sample_image_name = train_df["ID"].head().to_list()
sample_image_name

['image_1.jpg', 'image_2.jpeg', 'image_3.jpeg', 'image_4.jpeg', 'image_5.jpeg']

In [ ]:
import cv2

fig, axes = plt.subplots(1, len(sample_image_name), figsize=(15, 5))
i = 0
for img_name in sample_image_name:

    path = train_images_path + img_name
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    ax = axes[i]
    ax.axis("off")
    ax.imshow(img)
    ax.set_title(f"Image {i+1}")
    i = i + 1

plt.tight_layout()
plt.show()

In [ ]:
import os

imagefilenames = []
for dirname, _, filenames in os.walk(test_images_path):
    for filename in filenames:
        imagefilenames.append(os.path.join(dirname, filename))

In [ ]:
# Limit to first 5 images for visualization
num_images = min(5, len(imagefilenames))

fig, axes = plt.subplots(1, num_images, figsize=(15, 5))

for i in range(num_images):
    path = imagefilenames[i]
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    axes[i].imshow(img)
    axes[i].axis("off")  # Hide axis
    axes[i].set_title(f"Image {i+1}")

plt.show()

In [ ]:
species_labels = {species: i for i, species in enumerate(train_df["Species"].unique())}
species_names = {v: k for k, v in species_labels.items()}  # Reverse mapping
train_df["label"] = train_df["Species"].map(species_labels)

# Compute class weights for handling imbalance
class_weights = compute_class_weight(
    "balanced", classes=np.unique(train_df["label"]), y=train_df["label"]
)
class_weights_dict = {
    i: weight / np.max(class_weights) for i, weight in enumerate(class_weights)
}

In [ ]:
# best performer with train acc 98% and val 91% and score 0.914
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

# Data Augmentation for training
datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2,
)

# Training Data Generator
train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_images_path,
    x_col="ID",
    y_col="Species",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
)

# Validation Data Generator
val_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_images_path,
    x_col="ID",
    y_col="Species",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
)
num_classes = len(train_generator.class_indices)


# Load Pretrained MobileNetV2 model
base_model = tf.keras.applications.MobileNetV2(
    include_top=False, weights="imagenet", input_shape=(224, 224, 3)
)
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(512, activation="relu", kernel_regularizer=l2(0.0001))(x)
x = Dropout(0.5)(x)
out = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=out)


def cosine_annealing(epoch, lr):
    lr_min = 1e-6
    lr_max = 5e-5
    return lr_min + (lr_max - lr_min) * (1 + np.cos(np.pi * epoch / 30)) / 2


lr_scheduler = tf.keras.callbacks.LearningRateScheduler(cosine_annealing)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
model.compile(
    optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4),
    loss=loss,
    metrics=["accuracy"],
)

# Callbacks
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    lr_scheduler,
]

# Train Model
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weights_dict,
    callbacks=callbacks,
)

# Unfreeze some deeper layers and fine-tune
base_model.trainable = True
for layer in base_model.layers[:70]:
    layer.trainable = False

model.compile(
    optimizer=AdamW(learning_rate=2e-5, weight_decay=1e-4),
    loss=loss,
    metrics=["accuracy"],
)


model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    class_weight=class_weights_dict,
    callbacks=callbacks,
)

model.save("insect_classifier.h5")

print("Model training complete!")

Found 6873 validated image filenames belonging to 37 classes.
Found 1718 validated image filenames belonging to 37 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


215/215 ━━━━━━━━━━━━━━━━━━━━ 362s 2s/step - accuracy: 0.0507 - loss: 2.8342 - val_accuracy: 0.2445 - val_loss: 3.1381 - learning_rate: 5.0000e-05
Epoch 2/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 340s 2s/step - accuracy: 0.1939 - loss: 2.2799 - val_accuracy: 0.3871 - val_loss: 2.7281 - learning_rate: 4.9866e-05
Epoch 3/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 321s 1s/step - accuracy: 0.2928 - loss: 2.0119 - val_accuracy: 0.4773 - val_loss: 2.4222 - learning_rate: 4.9465e-05
Epoch 4/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 313s 1s/step - accuracy: 0.3703 - loss: 1.8418 - val_accuracy: 0.5140 - val_loss: 2.2113 - learning_rate: 4.8801e-05
Epoch 5/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 318s 1s/step - accuracy: 0.4275 - loss: 1.7161 - val_accuracy: 0.5623 - val_loss: 2.0997 - learning_rate: 4.7882e-05
Epoch 6/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 331s 2s/step - accuracy: 0.4813 - loss: 1.6334 - val_accuracy: 0.5920 - val_loss: 2.0293 - learning_rate: 4.6718e-05
Epoch 7/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 300s 1s/step - accuracy: 0.50

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
from tensorflow.keras.models import load_model

test_images_path = "/kaggle/input/hackathon-2-insect-species-classification/test/test/"
train_csv_path = "/kaggle/input/hackathon-2-insect-species-classification/train.csv"
model_path = "insect_classifier.h5"

train_df = pd.read_csv(train_csv_path)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

model = load_model(model_path)

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2,
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory="/kaggle/input/hackathon-2-insect-species-classification/train/train/",
    x_col="ID",
    y_col="Species",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
)

# Use the same label mapping from training
target_labels = train_generator.class_indices
species_names = {v: k for k, v in target_labels.items()}

# Get test image file names
test_images = os.listdir(test_images_path)
test_df = pd.DataFrame(
    {
        "ID": test_images,
        "filepath": [os.path.join(test_images_path, img) for img in test_images],
    }
)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col=None,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False,
)

# Predict on test images
predictions = model.predict(test_generator)
predicted_labels = np.argmax(predictions, axis=1)
predicted_species = [species_names[label] for label in predicted_labels]
submission_df = pd.DataFrame({"ID": test_images, "Species": predicted_species})
submission_df.to_csv("submission.csv", index=False)

print("Prediction completed and saved to submission.csv")